<a href="https://colab.research.google.com/github/zhengyangsu/Natural-Language-Processing/blob/main/latin_translation_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Latin to English Translation with LoRA

Adapted from the Week 1 tutorial `movie_review_sentiment_classifier_with_bert.ipynb`  
Use a GPU runtime

# 1. Setup Python Libraries (pip)

In [2]:
# transformers = models and tokenizers, peft = LoRA, bitsandbytes = 4-bit loading,
# accelerate = GPU placement, datasets = data loading, --quiet hides the long install log
!pip install -q transformers peft bitsandbytes accelerate datasets

# 2. Mount Google Drive

In [3]:
from google.colab import drive
import os, urllib.request


drive.mount('/content/drive')   # skip if already mounted

# Path to NLP_Project folder
project_path = '/content/drive/MyDrive/NLP_Project'

# Create directory if does not exist
if not os.path.exists(project_path):
    os.makedirs(project_path)
    print(f"Created directory: {project_path}")

# Change working directory
os.chdir(project_path)
print(f"Current working directory set to: {os.getcwd()}")

# Create the dataset folder (does nothing if it already exists)
os.makedirs('dataset', exist_ok=True)

# Download the custom test set
URL = 'https://raw.githubusercontent.com/zhengyangsu/Natural-Language-Processing/main/data/plautus_amphitruo_test50.jsonl'
DEST = 'dataset/plautus_amphitruo_test50.jsonl'
urllib.request.urlretrieve(URL, DEST)

# Check the download
print(os.listdir('dataset'))
with open(DEST, encoding='utf8') as f:
    print(sum(1 for _ in f), 'lines')   # should print 50


Mounted at /content/drive
Current working directory set to: /content/drive/MyDrive/NLP_Project
['plautus_amphitruo_test50.jsonl']
50 lines


# 3. Hugging Face Auth

In [4]:
#Will speed up download speed for Huggingface
from huggingface_hub import notebook_login
from huggingface_hub import whoami


!hf auth login
!hf auth whoami
#notebook_login()
#print(whoami()['name'])

Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
? How would you like to log in?  [Use arrows, Enter to confirm]
> Log in with your browser
  Paste an access token
? How would you like to log in? Log in with your browser

    Open this URL in your browser:
        https://hf.co/oauth/device

    And enter the code: PIRM-Y0VJ

    Waiting for authorization.......
Token is valid.
The token `oauth-zhengyangsu` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `oauth-zhengyangsu`
Note: This token will be refreshed automatically when it expires.
✓ Logged in
  user: zhengyangsu


# 4. Load the Dataset

In [7]:
from datasets import load_dataset
raw = load_dataset('grosenthal/latin_english_parallel')
mine = load_dataset('json', data_files={'test': 'dataset/plautus_amphitruo_test50.jsonl'})['test']

## Data Cleanup

In [8]:
!pip install langid --quiet # langid to check latin in en key

import re, unicodedata
from langid.langid import LanguageIdentifier, model
from collections import Counter

ident = LanguageIdentifier.from_modelstring(model, norm_probs=True)
drop_counts = Counter()   # counts per reason, reset before each split


# Unicode normalize, remove html tags, collapse space
def clean_text(s):
    s = unicodedata.normalize('NFC', s)   # make accented letters use one consistent form
    s = re.sub(r'<[^>]+>', ' ', s)        # remove tags such as <br />
    return ' '.join(s.split())            # collapse whitespace

# True if en column is 0.9 confident detected as Latin
def en_is_latin(x):
    if len(x['en'].split()) < 6:          # skip short text
        return False
    lang, prob = ident.classify(x['en'])
    return lang == 'la' and prob > 0.9

# Keep pair only if it passes all checks
def is_ok(x):
    n_la, n_en = len(x['la'].split()), len(x['en'].split())
    if n_la < 3 or n_en < 3:              # sentence too short
        drop_counts['short_sentence'] += 1
        return False
    if en_is_latin(x):                    # en column is latin
        drop_counts['en_is_latin'] += 1
        return False
    if not (0.3 <= n_en / n_la <= 4):     # lengths misalignment
        drop_counts['length_misalignment'] += 1
        return False
    return True

# Text cleaning
raw = raw.map(lambda x: {'la': clean_text(x['la']), 'en': clean_text(x['en'])})

# Filter splits with the same rule
for k in ['train', 'valid', 'test']:
    drop_counts.clear() # reset split filter count
    before = len(raw[k])
    raw[k] = raw[k].filter(is_ok, load_from_cache_file=False)   # force counters update
    print(f"\n[{k}] {before} -> {len(raw[k])}")
    for reason in ['short_sentence', 'en_is_latin', 'length_misalignment']:
        print(f"  {reason:20s}: {drop_counts[reason]}")
    print(f"  {'total dropped':20s}: {sum(drop_counts.values())}")

Filter:   0%|          | 0/99343 [00:00<?, ? examples/s]


[train] 99343 -> 98225
  short_sentence      : 353
  en_is_latin         : 736
  length_misalignment : 29
  total dropped       : 1118


Filter:   0%|          | 0/1014 [00:00<?, ? examples/s]


[valid] 1014 -> 1002
  short_sentence      : 1
  en_is_latin         : 10
  length_misalignment : 1
  total dropped       : 12


Filter:   0%|          | 0/1014 [00:00<?, ? examples/s]


[test] 1014 -> 996
  short_sentence      : 5
  en_is_latin         : 12
  length_misalignment : 1
  total dropped       : 18


## Add EOS and Cap MAX_LEN

In [6]:
from transformers import AutoTokenizer

checkpoint = 'hathibelagal/llama-3.2-latin'  #base model
tokenizer = AutoTokenizer.from_pretrained(checkpoint, clean_up_tokenization_spaces=False)
tokenizer.pad_token = tokenizer.eos_token

MAX_LEN = 256
EOS = tokenizer.eos_token

def build_text(la, en):
    return f"Latin: {la}\nEnglish: {en}{EOS}"           # add EOS

# remove beyond MAX_LEN
def fits(batch):
    texts = [build_text(la, en) for la, en in zip(batch['la'], batch['en'])]
    return [len(ids) <= MAX_LEN for ids in tokenizer(texts)['input_ids']]

for k in ['train', 'valid', 'test']:
    before = len(raw[k])
    raw[k] = raw[k].filter(fits, batched=True, batch_size=1000)
    print(k, before, '->', len(raw[k]))

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

Filter:   0%|          | 0/98225 [00:00<?, ? examples/s]

train 98225 -> 96455


Filter:   0%|          | 0/1002 [00:00<?, ? examples/s]

valid 1002 -> 991


Filter:   0%|          | 0/996 [00:00<?, ? examples/s]

test 996 -> 974


## Check for Overlapping

In [30]:
N_TRAIN = 2000
N_VAL   = 200
N_TEST  = 200
SEED    = 42

# Only used for comparing sentences, never changes the data
def key(s):
    return s.lower()

# 1. Check split overlapping
full_train_la = {key(x) for x in raw['train']['la']}
print('test/train overlap:', sum(key(x) in full_train_la for x in raw['test']['la']), '/', len(raw['test']))
print('valid/train overlap:', sum(key(x) in full_train_la for x in raw['valid']['la']), '/', len(raw['valid']))

# val/test set size
val = raw['valid'].select(range(N_VAL))
val_la = {key(x) for x in val['la']}
# take test rows in val
test = raw['test'].filter(lambda x: key(x['la']) not in val_la).select(range(N_TEST))

# remove val/test sentences from train
held_la = {key(x) for x in val['la']} | {key(x) for x in test['la']}
held_en = {key(x) for x in val['en']} | {key(x) for x in test['en']}

pool = raw['train'].filter(
    lambda x: key(x['la']) not in held_la and key(x['en']) not in held_en
)
print('train pool:', len(raw['train']), '->', len(pool))

# remove duplicate Latin sentences inside train
seen = set()
def first_time(x):
    k = key(x['la'])
    if k in seen:
        return False
    seen.add(k)
    return True
pool = pool.filter(first_time)
print('unique:', len(pool))

dataset = {
    'train': pool.shuffle(seed=SEED).select(range(N_TRAIN)),
    'val':   val,
    'test':  test,
}

# verification
train_la = {key(x) for x in dataset['train']['la']}
print('train ∩ test:', sum(key(x) in train_la for x in dataset['test']['la']))
print('train ∩ val :', sum(key(x) in train_la for x in dataset['val']['la']))
print('val ∩ test  :', len({key(x) for x in dataset['val']['la']} & {key(x) for x in dataset['test']['la']}))

test/train overlap: 7 / 974
valid/train overlap: 6 / 991


Filter:   0%|          | 0/974 [00:00<?, ? examples/s]

Filter:   0%|          | 0/96455 [00:00<?, ? examples/s]

train pool: 96455 -> 96399


Filter:   0%|          | 0/96399 [00:00<?, ? examples/s]

unique: 96120
train ∩ test: 0
train ∩ val : 0
val ∩ test  : 0


##Dataframe Display

In [31]:
import pandas as pd

print({k: len(v) for k, v in raw.items()})
print(dataset['train'].column_names)

pd.set_option('display.max_colwidth', None)

dataset['train'].to_pandas().head()

from IPython.display import display

for name in ['train', 'val', 'test']:
    print(name)
    display(dataset[name].to_pandas().head(3))

{'train': 96455, 'test': 974, 'valid': 991}
['id', 'la', 'en', 'file']
train


,id,la,en,file
0,52883,Spatium velut aequoris electum quo Mosae fluminis os amnem Rhenum Oceano adfundit.,"The place chosen for the display was a small sea, so to speak, formed at the point where the mouth of the Maas discharges the water of the Rhine into the ocean.",final_alignments\Tacitus_Histories_Book5.json
1,14680,Futurum esse paucis annis uti omnes ex Galliae finibus pellerentur atque omnes Germani Rhenum transirent; neque enim conferendum esse Gallicum cum Germanorum agro neque hanc consuetudinem victus cum illa comparandam.,"The result would be that within a short time, all of them would be expelled from the regions of Gaul, and every German would cross the Rhine; because the land of Gaul must not be compared to that of the Germans, neither should the way of life of the latter be equated with that of the former.",final_alignments\Caesar_DBG_Book1.json
2,37160,"simplicitatem columbae in ecclesia servat, in foro serpentis astutiam; bonis prudens, malis cautus, neutris callidus iudicatur.","In the church he retains the innocence of a dove, in courts the intelligence of a serpent. He appears prudent to the virtuous, wary to the wicked, and artful to neither.",final_alignments\Sidonius_Letters_Book7.json


val


,id,la,en,file
0,36149,"nam memor originis tuae nobilium sibi persuasit universitas, quamdiu nos Sabini familia rexerit, Sabiniani familiam non timendam.","For the entire aristocracy, who recall the lineage that you belong to, are certain that as long as the descendants of Sabinus are our leaders, there is no reason for us to be afraid of the descendants of Sabinianus.",final_alignments\Sidonius_Letters_Book3.json
1,27252,"Tanta tibi bene faciendi vis, ut indulgentia tua necessitates aemuletur.",Such is your ability to perform well that your merciful kindness has an equivalent impact as harsh compulsion.,final_alignments\Pliny_Younger_Panegyricus.json
2,28719,"tempore nec senior Pater est, nec numine maior, nam sapiens retro semper Deus edidit ex se, per quod semper erat, gignenda ad saecula Verbum.","The Father is not older in time nor greater in divinity because God has been wise throughout infinite time and released from Himself the Word to create the world, through which he always existed.",final_alignments\Prudentius_Divinity_Of_Christ.json


test


,id,la,en,file
0,32668,"Tibi autem, qui sapis, quam potest denuntio ipsi mihi indices quid ex eis cognoveris, ne possis infitiari dixisse, quae dixeris.","But I offer you a serious caution that you tell me directly what you have gained from those sacrifices, since I know, so that you will not be able to deny having spoken to me.",final_alignments\Quintus_Curtius_Alexander_Book7.json
1,90310,currens autem vir de Beniamin ex acie venit in Silo in die illo scissa veste et conspersus pulvere caput,"And there ran a man of Benjamin out of the army, and came to Silo the same day, with his clothes rent, and his head strewed with dust.",final_alignments\Vulgate_Bible.json
2,16191,"Exposuit quid iniquitas loci posset, quid ipse ad Avaricum sensisset, cum sine duce et sine equitatu deprehensis hostibus exploratam victoriam dimisisset, ne parvum modo detrimentum in contentione propter iniquitatem loci accideret.","""And he said what disadvantage the ground could have on the outcome, recalling his own opinion at Avaricum where, despite having surprised the enemy without the aid of a general or cavalry, he abandoned a certain victory in order to avoid even a small loss caused by the disadvantage of the position.""",final_alignments\Caesar_DBG_Book7.json


# Tokenize the Dataset

In [32]:
def tokenization(batch):
    texts = [build_text(la, en) for la, en in zip(batch['la'], batch['en'])]
    return tokenizer(texts, truncation=True, max_length=MAX_LEN)

tokenized = {k: v.map(tokenization, batched=True, remove_columns=v.column_names)
             for k, v in dataset.items()}

assert all(ids[-1] == tokenizer.eos_token_id for ids in tokenized['train']['input_ids'][:100]), 'EOS is missing'


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

# Load the Model and Add LoRA

In [33]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True,  # store the frozen base weights in 4 bits
                         bnb_4bit_quant_type='nf4',  # NF4 is the 4-bit format designed for QLoRA
                         bnb_4bit_compute_dtype=torch.float16)  # compute in 16-bit floats

model = AutoModelForCausalLM.from_pretrained(checkpoint,  # download base model
                                             quantization_config=bnb,  # apply the 4-bit settings above
                                             device_map='auto')  # training on the GPU automatically

model = prepare_model_for_kbit_training(
    model,
    gradient_checkpointing_kwargs={'use_reentrant': False})  # casts some layers to full precision, cant update 4bit weight

model = get_peft_model(model, LoraConfig(r=16,  # rank of the LoRA matrices: higher = more capacity
                                         lora_alpha=32,  # scaling factor for the LoRA update
                                         target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],  # add LoRA to the attention layers
                                         task_type='CAUSAL_LM'))  # word-prediction with Parameter Efficient Fine Tuning（
model.print_trainable_parameters()  # show how few parameters are actually trained

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

trainable params: 9,175,040 || all params: 3,221,924,864 || trainable%: 0.2848


# Setup Training Configurations

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling  # training tools

class CausalLMCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.tokenizer.padding_side = 'right'

    def __call__(self, features):
        self.tokenizer.padding_side = 'right'
        batch = self.tokenizer.pad(features, return_tensors='pt')
        labels = batch['input_ids'].clone()
        labels[batch['attention_mask'] == 0] = -100 #ignore padding，keep EOS
        batch['labels'] = labels
        return batch

training_args = TrainingArguments(
    seed=42,  # fixed random seed
    output_dir='./results',  # folder for chkpts
    num_train_epochs=1,  # total passes
    per_device_train_batch_size=4,  # 4 examples per step (small to fit GPU memory)
    gradient_accumulation_steps=4,  # accumulate 4 steps before each update = effective batch size 16
    learning_rate=2e-4,  # step size
    fp16=True,  # 16-bit training (T4 supports fp16 not bf16)
    eval_strategy='epoch',  # evaluate on the validation set at the end of each epoch
    save_strategy='epoch',
    load_best_model_at_end=True,   # load best chkpt
    greater_is_better=False,
    logging_strategy="steps",
    logging_steps=10,
    report_to='none',  # do not log
)

trainer = Trainer(
    model=model,  # LoRA model
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['val'],
    data_collator=CausalLMCollator(tokenizer)
)

# Define evaluation

In [ ]:
# Install the metric scorers: sacrebleu (BLEU/chrF) and rouge_score (ROUGE)
!pip install -q sacrebleu evaluate rouge_score
import json, numpy as np, torch, evaluate
from tqdm.auto import tqdm   # progress bar

# Load the translation metrics
bleu  = evaluate.load('sacrebleu')   # word n-gram overlap with the reference
chrf  = evaluate.load('chrf')        # character n-gram overlap, more forgiving for word endings
rouge = evaluate.load('rouge')       # recall-oriented overlap (ROUGE-1/2/L), used here as a supporting metric

def translate_batch(latin_list, max_new_tokens=120):
    """Translate a list of Latin sentences in one batch and return the English outputs."""
    tokenizer.padding_side = 'left'   # generation appends tokens on the right, so pad on the left
    try:
        # Same prompt format as training, but stop after "English:" so the model writes the translation
        prompts = [f"Latin: {la}\nEnglish:" for la in latin_list]

        # Convert text to token ids; pad to equal length and cap the prompt at MAX_LEN tokens
        inputs = tokenizer(prompts, return_tensors='pt', padding=True,
                           truncation=True, max_length=MAX_LEN).to(model.device)

        with torch.no_grad():   # no gradients needed for inference, saves memory
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False,                       # greedy decoding, same output every run
                                 pad_token_id=tokenizer.eos_token_id)   # avoid the pad_token_id warning

        # Drop the prompt tokens and decode only the newly generated part
        texts = tokenizer.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        # Keep only the first line, in case the model keeps writing after the translation
        return [t.split('\n')[0].strip() for t in texts]
    finally:
        tokenizer.padding_side = 'right'   # restore right padding for training, even if generation failed

def full_eval(label, batch_size=8):
    """Report loss, perplexity, BLEU, chrF and ROUGE, and save predictions. `label` names the output files."""
    model.eval()   # switch off dropout and other training-only behavior
    print(f'===== {label} =====')

    # 1. Loss and perplexity on the tokenized test set
    #    (only the dataset test split, because the custom test set `mine` is not tokenized)
    result = trainer.evaluate(tokenized['test'])
    print(f"test loss {result['eval_loss']:.4f} | perplexity {np.exp(result['eval_loss']):.2f}")

    # 2. Translate each test set, then score the translations against the references
    for name, split in [('test', dataset['test']), ('mine', mine)]:   # 'mine' = custom Perseus test set
        preds = []
        for i in tqdm(range(0, len(split), batch_size), desc=name):
            preds += translate_batch(split['la'][i:i + batch_size])   # translate one batch at a time

        refs_nested = [[r] for r in split['en']]   # sacrebleu / chrF expect a list of reference lists
        refs_flat   = list(split['en'])            # ROUGE takes one reference string per prediction

        b = bleu.compute(predictions=preds, references=refs_nested)['score']
        c = chrf.compute(predictions=preds, references=refs_nested)['score']
        r = rouge.compute(predictions=preds, references=refs_flat)   # returns 0-1 scores

        # Multiply ROUGE by 100 so it is on the same 0-100 scale as BLEU and chrF
        print(f"{name:5s} | BLEU {b:.2f} | chrF {c:.2f} | "
              f"ROUGE-1 {r['rouge1']*100:.2f} | ROUGE-2 {r['rouge2']*100:.2f} | ROUGE-L {r['rougeL']*100:.2f}")

        # Save Latin / reference / prediction so error analysis does not require re-running the model
        with open(f'preds_{label}_{name}.jsonl', 'w', encoding='utf-8') as f:
            for la, ref, pred in zip(split['la'], split['en'], preds):
                f.write(json.dumps({'la': la, 'ref': ref, 'pred': pred}, ensure_ascii=False) + '\n')

Training Loss,Validation Loss,Epoch
No log,3.219894,0


{'eval_loss': 3.2198944091796875} perplexity = 25.0254775811166


# Evaluate the Un-Fine-Tuned Model for a Baseline

In [ ]:
# Baseline run before fine-tuning; call full_eval('finetuned') after training
full_eval('baseline')

# Fine-Tune with LoRA

In [ ]:
trainer.train()  # run the LoRA fine tuning loop, Traning loss loging step is 500

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,No log,2.793176


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=125, training_loss=2.798360107421875, metrics={'train_runtime': 589.5015, 'train_samples_per_second': 3.393, 'train_steps_per_second': 0.212, 'total_flos': 4999518002601984.0, 'train_loss': 2.798360107421875, 'epoch': 1.0})

# Evaluate After Fine-Tuning

In [ ]:
# Evaluate After Fine Tuning
full_eval('finetuned')

Training Loss,Validation Loss,Epoch
No log,2.794720,1


{'eval_loss': 2.794720411300659} perplexity = 16.358054586538767


# Try out some examples and save the adapter

In [22]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

def translate(la, beams=1, max_new_tokens=120):
    out = pipe(f"Latin: {la}\nEnglish:",
               max_new_tokens=max_new_tokens,
               do_sample=False,
               num_beams=beams,                  # 1 = greedy，4 = beam search
               repetition_penalty=1.1 if beams > 1 else 1.0,
               pad_token_id=tokenizer.eos_token_id,   # avoid "Setting pad_token_id" warning
               return_full_text=False)
    return out[0]['generated_text'].split('\n')[0].strip()

q = "Quo usque tandem abutere, Catilina, patientia nostra?"
print('greedy:', translate(q))
print('beam=4:', translate(q, beams=4))

model.save_pretrained('latin_lora')  # save LoRA adapter

NameError: name 'tokenizer' is not defined

# Reuse LORA

In [ ]:
'''from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(checkpoint, quantization_config=bnb, device_map='auto')
model = PeftModel.from_pretrained(base, 'latin_lora')'''